In [1]:
#Getting all the thursday votes

In [1]:
import os
import time
import json
from pathlib import Path
import pandas as pd
import glob
import numpy as np
import pprint
import requests
from datetime import datetime
headers = {"User-Agent": "MyDataResearchProject-dev-1.0.0", "Accept": "application/ld+json"}

In [2]:
df = pd.read_csv("all_thursdays.csv")

In [3]:
df.head()

,Unnamed: 0,id,type,activity_date,activity_end_date,activity_id,activity_label,activity_start_date,consists_of,documented_by_a_realization_of,had_activity_type,had_excused_person,had_participant_person,number_of_attendees,parliamentary_term,recorded_in_a_realization_of,hasLocality,was_scheduled_in
0,137,eli/dl/event/MTG-PL-2014-01-16,Activity,"{'@value': '2014-01-16T00:00:00+01:00', 'type'...",2014-01-16T23:00:00+01:00,MTG-PL-2014-01-16,"{'lv': 'Ceturtdiena, 2014. gada 16. janvāris',...",2014-01-16T01:00:00+01:00,['eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-34472...,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-7,NaN,http://publications.europa.eu/resource/authori...,NaN
1,138,eli/dl/event/MTG-PL-2014-02-06,Activity,"{'@value': '2014-02-06T00:00:00+01:00', 'type'...",2014-02-06T23:00:00+01:00,MTG-PL-2014-02-06,"{'lv': 'Ceturtdiena, 2014. gada 6. februāris',...",2014-02-06T01:00:00+01:00,['eli/dl/event/MTG-PL-2014-02-06-VOT-ITM-34584...,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-7,NaN,http://publications.europa.eu/resource/authori...,NaN
2,139,eli/dl/event/MTG-PL-2014-02-27,Activity,"{'@value': '2014-02-27T00:00:00+01:00', 'type'...",2014-02-27T23:00:00+01:00,MTG-PL-2014-02-27,"{'hr': 'četvrtak, 27. veljače 2014.', 'de': 'D...",2014-02-27T01:00:00+01:00,['eli/dl/event/MTG-PL-2014-02-27-VOT-ITM-34747...,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-7,NaN,http://publications.europa.eu/resource/authori...,NaN
3,140,eli/dl/event/MTG-PL-2014-03-13,Activity,"{'@value': '2014-03-13T00:00:00+01:00', 'type'...",2014-03-13T23:00:00+01:00,MTG-PL-2014-03-13,"{'sk': 'Štvrtok, 13. marca 2014', 'de': 'Donne...",2014-03-13T01:00:00+01:00,['eli/dl/event/MTG-PL-2014-03-13-VOT-ITM-34747...,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-7,NaN,http://publications.europa.eu/resource/authori...,NaN
4,141,eli/dl/event/MTG-PL-2014-04-03,Activity,"{'@value': '2014-04-03T00:00:00+02:00', 'type'...",2014-04-03T23:00:00+02:00,MTG-PL-2014-04-03,"{'lv': 'Ceturtdiena, 2014. gada 3. aprīlis', '...",2014-04-03T01:00:00+02:00,['eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-35084...,NaN,def/ep-activities/PLENARY_SITTING,NaN,NaN,NaN,org/ep-7,NaN,http://publications.europa.eu/resource/authori...,NaN


In [4]:
#We start by getting all the vote results for each row
#first we clean column "id"
df["id"] = df["id"].str.replace("eli/dl/event/", "")

In [5]:
#Making the consists of column into a list
#We might not need these though
import ast
first_row_items = df["consists_of"].iloc[0]

if isinstance(first_row_items, str):
    first_row_items = ast.literal_eval(first_row_items)
    
for event_id in first_row_items:
        
        if "VOT" in event_id:
            print(event_id)

eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344722-6
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344715-2
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344717-7
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344715-3
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345248-1
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344721-9
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345210-10
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345249-4
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345351-12
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344718-8
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345350-14
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-344724-11
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345211-5
eli/dl/event/MTG-PL-2014-01-16-VOT-ITM-345348-13


In [ ]:
#Finding the legislative bills voted on thursdays which were in the second reading
#since the way the EU manages data has changed over the past decade, we will have to search for different keywords
#in both french and english
found_second_readings = []
second_readings_json_folder = "raw_meetings_jsons"
os.makedirs(second_readings_json_folder, exist_ok=True)

# 1. Explicitly define your required headers, plus the language preference
headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0", 
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

# Define our magic search words (covering both new formatting and older French/English text)
#we make all search terms lowercase to make sure we don't miss anything cus of capitalization issues
second_reading_keywords = ["***ii", "*** ii", "second reading", "deuxième lecture", "deuxieme lecture", "deuxi\u00e8me lecture"]

for meeting_id in df["id"]:
    print(f"Scanning meeting: {meeting_id}...")
    url = f"https://data.europarl.europa.eu/api/v2/meetings/{meeting_id}/vote-results"
    
    try:
        # 2. Pass your master headers directly into the request
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()

            # Save the json dump
            try:
                # Sanitize the ID because it contains slashes (eli/dl/event/...)
                safe_filename = str(meeting_id).replace("/", "_") + ".json"
                file_path = os.path.join(second_readings_json_folder, safe_filename)
                with open(file_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, indent=4)
            except Exception as save_err:
                print(f"  -> Warning: Could not save file for {meeting_id}: {save_err}")
            
            if "data" in data and isinstance(data["data"], list):
                
                for vote_item in data["data"]:
                    
                    # Grab everything and convert to a lowercase string for easy searching
                    labels_str = str(vote_item.get("structuredLabel", "")).lower() + str(vote_item.get("activity_label", "")).lower()
                    
                    # Check if ANY of our magic keywords exist in the text
                    if any(keyword in labels_str for keyword in second_reading_keywords):
                        
                        item_id = vote_item.get("id", "Unknown ID")
                        
                        # Smart Title Grabber: Try English first, then fallback to French, then just grab whatever is there
                        final_title = "Title not found"
                        structured_label = vote_item.get("structuredLabel", {})
                        
                        if isinstance(structured_label, dict):
                            if "en" in structured_label:
                                final_title = structured_label["en"]
                            elif "fr" in structured_label:
                                final_title = structured_label["fr"]
                            elif len(structured_label) > 0:
                                # Just grab the first available translation
                                first_key = list(structured_label.keys())[0]
                                final_title = structured_label[first_key]
                        
                        print(f"\n🚨 FOUND ONE! Meeting: {meeting_id}")
                        print(f"ID: {item_id}")
                        print(f"Title: {final_title}\n")
                        
                        found_second_readings.append({
                            "meeting_id": meeting_id,
                            "vot_itm_id": item_id,
                            "title": final_title
                        })
                        
        elif response.status_code == 204:
            pass # Silently skip empty days to keep the terminal clean
        else:
            print(f"Failed to fetch {meeting_id}: Status {response.status_code}")
            
    except Exception as e:
        print(f"Error on {meeting_id}: {e}")
        
    time.sleep(0.5)

df_second_readings = pd.DataFrame(found_second_readings)

print(f"\n--- SCAN COMPLETE ---")
print(f"Total Second Readings found: {len(df_second_readings)}")
if not df_second_readings.empty:
    print(df_second_readings.head())

Scanning meeting: MTG-PL-2014-01-16...
Scanning meeting: MTG-PL-2014-02-06...
Scanning meeting: MTG-PL-2014-02-27...
Scanning meeting: MTG-PL-2014-03-13...
Scanning meeting: MTG-PL-2014-04-03...

🚨 FOUND ONE! Meeting: MTG-PL-2014-04-03
ID: eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64
Title: <structuredLabel><label>Recommandation pour la deuxième lecture: Ria Oomen-Ruijten (A7-0188/2014)</label></structuredLabel>


🚨 FOUND ONE! Meeting: MTG-PL-2014-04-03
ID: eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350903-1
Title: <structuredLabel><title>Régime communautaire de contrôle des exportations, des transferts, du courtage et du transit de biens à double usage</title><label>Recommandation pour la deuxième lecture: Christofer Fjellner (A7-0236/2014)</label></structuredLabel>

Scanning meeting: MTG-PL-2014-04-17...
Scanning meeting: MTG-PL-2014-07-03...
Scanning meeting: MTG-PL-2014-07-17...
Scanning meeting: MTG-PL-2014-09-18...
Scanning meeting: MTG-PL-2014-10-23...
Scanning meeting: MTG-PL-2

In [3]:
#We are finding some second reading votes but there might be many that we are missing.
#In order to see if we should be searching for different things we will do a spot check
# the bill 2018/0152B(COD) was a second reading. we will check what the json structure
#for that bill looks like. We are searching through 2021 records because we know that the bill was voted on in 2021. We will check if we can find it in the 2021 records.

headers = {
    "User-Agent": "MyDataResearchProject-dev-1.0.0",
    "Accept": "application/ld+json",
    "Accept-Language": "en,fr;q=0.9"
}

print("Fetching 2021 calendar data...")
calendar_url = "https://data.europarl.europa.eu/api/v2/meetings"
calendar_params = {
    "year": "2021",
    "format": "application/ld+json"
}

save_folder = "vote_results_2021_jsons"
os.makedirs(save_folder, exist_ok=True)

# The variations we established
target_variations = ["2018-0152", "2018/0152"] 
found = False

try:
    calendar_resp = requests.get(calendar_url, headers=headers, params=calendar_params)
    if calendar_resp.status_code == 200:
        calendar_data = calendar_resp.json()
        meetings = calendar_data.get("data", [])
        print(f"Found {len(meetings)} meetings in 2021. Commencing hunt for 2018/0152B...")
        
        # 2. Loop through all 2021 meetings
        for meeting in meetings:
            
            # THE FIX: Clean the ID immediately so the URL formats correctly!
            raw_id = meeting.get("id", "")
            clean_id = raw_id.replace("eli/dl/event/", "")
            
            print(f"Scanning {clean_id}...")
            
            # 3. Hit the deep vote-results layer
            vote_url = f"https://data.europarl.europa.eu/api/v2/meetings/{clean_id}/vote-results"
            
            try:
                vote_resp = requests.get(vote_url, headers=headers)
                
                if vote_resp.status_code == 200:
                    vote_data = vote_resp.json()

                    try:
                        safe_filename = f"{clean_id}.json"
                        file_path = os.path.join(save_folder, safe_filename)
                        with open(file_path, "w", encoding="utf-8") as f:
                            json.dump(vote_data, f, indent=4)
                    except Exception as save_err:
                        print(f"  -> Warning: Could not save file for {clean_id}: {save_err}")

                    raw_text = vote_resp.text
                    
                    if any(target in raw_text for target in target_variations):
                        print(f"\n🚨 BINGO! Trapped the ghost procedure in meeting: {clean_id} 🚨\n")
                        
                        vote_data = vote_resp.json()
                        for item in vote_data.get("data", []):
                            item_str = json.dumps(item)
                            
                            if any(target in item_str for target in target_variations):
                                print("=== RAW VOTE ITEM JSON ===")
                                print(json.dumps(item, indent=4))
                        
                        found = True
                        break # Stop searching
                        
            except Exception as e:
                print(f"  -> Error checking {clean_id}: {e}")
                
            if found:
                break
            
            time.sleep(0.5) # Respect API limits
            
except Exception as e:
    print(f"Failed to fetch calendar: {e}")

if not found:
    print("\n❌ Could not find the procedure in 2021... the plot thickens!")

Fetching 2021 calendar data...
Found 55 meetings in 2021. Commencing hunt for 2018/0152B...
Scanning MTG-PL-2021-01-18...
Scanning MTG-PL-2021-01-19...
Scanning MTG-PL-2021-01-20...
Scanning MTG-PL-2021-01-21...
Scanning MTG-PL-2021-02-08...
Scanning MTG-PL-2021-02-09...
Scanning MTG-PL-2021-02-10...
Scanning MTG-PL-2021-02-11...
Scanning MTG-PL-2021-03-08...
Scanning MTG-PL-2021-03-09...
Scanning MTG-PL-2021-03-10...
Scanning MTG-PL-2021-03-11...
Scanning MTG-PL-2021-03-24...
Scanning MTG-PL-2021-03-25...
Scanning MTG-PL-2021-04-26...
Scanning MTG-PL-2021-04-27...
Scanning MTG-PL-2021-04-28...
Scanning MTG-PL-2021-04-29...
Scanning MTG-PL-2021-05-17...
Scanning MTG-PL-2021-05-18...
Scanning MTG-PL-2021-05-19...
Scanning MTG-PL-2021-05-20...
Scanning MTG-PL-2021-05-21...
Scanning MTG-PL-2021-06-07...
Scanning MTG-PL-2021-06-08...
Scanning MTG-PL-2021-06-09...
Scanning MTG-PL-2021-06-10...
Scanning MTG-PL-2021-06-23...
Scanning MTG-PL-2021-06-24...
Scanning MTG-PL-2021-07-05...
Scanning

In [ ]:
# The spot check above did not find the case we were looking for, because the procedure code was not in the json
#However, looking into the json file for the day of that vote shows that it was described as "deuxième lecture"
#Which means that our original second readings finding code would have caught it
# This means we can trust the data of second_readings_found.csv

In [8]:
df_second_readings.sort_values("meeting_id", ascending=True)

,meeting_id,vot_itm_id,title
0,MTG-PL-2014-04-03,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350882-64,<structuredLabel><label>Recommandation pour la...
1,MTG-PL-2014-04-03,eli/dl/event/MTG-PL-2014-04-03-VOT-ITM-350903-1,<structuredLabel><title>Régime communautaire d...
2,MTG-PL-2016-04-14,eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522221-1,<structuredLabel><title>Protection des personn...
3,MTG-PL-2016-04-14,eli/dl/event/MTG-PL-2016-04-14-VOT-ITM-522222-2,<structuredLabel><title>Protection des personn...
4,MTG-PL-2016-04-28,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520211-8,<structuredLabel><title>Interopérabilité du sy...
5,MTG-PL-2016-04-28,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520212-9,<structuredLabel><title>Sécurité ferroviaire</...
6,MTG-PL-2016-04-28,eli/dl/event/MTG-PL-2016-04-28-VOT-ITM-520210-7,<structuredLabel><title>Agence de l’Union euro...
7,MTG-PL-2016-06-09,eli/dl/event/MTG-PL-2016-06-09-VOT-ITM-540102-4,<structuredLabel><title>Favoriser la libre cir...
9,MTG-PL-2025-10-23,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975337,<structuredLabel><title>Preventing plastic pel...
8,MTG-PL-2025-10-23,eli/dl/event/MTG-PL-2025-10-23-VOT-ITM-975331,<structuredLabel><title>Soil Monitoring and Re...


In [9]:
df_second_readings.to_csv("second_readings_found.csv", index=False)

In [ ]:
#We make a loop to create nested api calls
all_extracted_thursdays = []
for meeting_id in df["id"]:
    url_1 = f"https://data.europarl.europa.eu/api/v2/meetings/{meeting_id}/vote-results"
    
    try:
        response_1 = requests.get(url_1, headers=headers)
        if response_1.status_code == 200:
            data_1 = response_1.json()

            #save the json dump
            safe_filename = f"{meeting_id}.json"
            file_path = os.path.join(meetings_json_folder, safe_filename)

            if "data" in data_1 and isinstance(data_1["data"], list):
                for vote_item in data_1["data"]:
                    if "consists_of" in vote_item and isinstance(vote_item["consists_of"], list):
                        decision_ids = vote_item["consists_of"]

                        for decision_id in decision_ids:
                            if "-DEC-" not in decision_id:
                                continue
                                
                            url_2 = f"https://data.europarl.europa.eu/{decision_id}"
                            response_2 = requests.get(url_2, headers=headers)

                            if response_2.status_code == 200:
                                decision_data = response_2.json()

                                #save the json dump
                                safe_filename = decision_id.split("/")[-1] + ".json"
                                file_path = os.path.join(decisions_json_folder, safe_filename)
                                
                                # Write the JSON data to the file
                                with open(file_path, "w", encoding="utf-8") as f:
                                    json.dump(decision_data, f, indent=4)

                                if "data" in decision_data and len(decision_data["data"]) > 0:
                                    target_dict = decision_data["data"][0]

                                    record = {
                                        "meeting_id": meeting_id,
                                        "decision_id": decision_id,
                                        "start_date": target_dict.get("activity_start_date"),
                                        "method": target_dict.get("decision_method"),
                                        "outcome": target_dict.get("decision_outcome"),
                                        "attendees": target_dict.get("number_of_attendees"),
                                        "votes_favor": target_dict.get("number_of_votes_favor"),
                                        "votes_against": target_dict.get("number_of_votes_against")
                                    }
                                    all_extracted_thursdays.append(record)

                            else:
                                print(f"Failed to fetch decision {decision_id}: {response_2.status_code}")

                            time.sleep(0.5)

            else:
                print(f"No valid 'data' list found for meeting {meeting_id}")
        else:
            print(f"Failed to fetch meeting {meeting_id}: {response_1.status_code}")
    except Exception as e:
        print(f"An error occurred while processing {meeting_id}: {e}")
        time.sleep(0.5)

df_extracted_thursdays = pd.DataFrame(all_extracted_thursdays)

In [ ]:
df_extracted_thursdays.head()